# Gradient Boosting - Day 2 (Scikit-Learn)

## Loss Functions and Gradient

In regression problems, it makes sense to define the signed error as the difference between the prediction and the label. However, in other types of problems this strategy often leads to poor results.

A better strategy used in gradient boosting is to:

1. Define a loss function similar to the loss functions used in neural networks (e.g., entropy/log loss for classification)
2. Train the weak model to predict the **gradient** of the loss according to the strong model output

---

## Pseudo Response Formula

Formally, given a loss function $L(y, p)$ where $y$ is a label and $p$ a prediction, the pseudo response used to train the weak model at step $i$ is:

$$\tilde{y} = -\left[ \frac{\partial L(y, F(x))}{\partial F(x)} \right]_{F(x)=F_{i-1}(x)}$$

where $F(x)$ is the prediction of the strong model.

---

## Regression with Squared Error

For regression, squared error is a common loss function:

$$L(y, F) = \frac{1}{2}(y - F)^2$$

The gradient is:

$$\frac{\partial L}{\partial F} = F - y$$

So the pseudo response becomes:

$$\tilde{y} = y - F$$

In other words, the gradient is the signed error. Note that constant factors do not matter because of shrinkage.

**Important:** This equivalence is only true for regression problems with squared error loss. For other problems (classification, ranking, regression with percentile loss), there is no equivalence between gradient and signed error.

---

## Classification with Log Loss (Entropy)

For binary classification, the log loss (entropy) is:

$$L(y, p) = -y \log(p) - (1-y) \log(1-p)$$

The gradient is different from simple signed error, which is why gradient boosting uses the gradient directly.

---

## Newton's Method in Gradient Boosting

Newton's method is an optimization method like gradient descent. However, unlike gradient descent which only uses the gradient (first derivative), Newton's method uses both:
- First derivative (gradient)
- Second derivative (Hessian)

| Method | Formula | Information Used |
|--------|---------|------------------|
| Gradient Descent | $x_{n+1} = x_n - \eta \nabla f(x_n)$ | First derivative only |
| Newton's Method | $x_{n+1} = x_n - \eta \frac{\nabla f(x_n)}{\nabla^2 f(x_n)}$ | First and second derivatives |

---

## Newton's Method in Gradient Boosted Trees

Newton's method can be integrated in two ways:

1. **Leaf Value Optimization:** Once a tree is trained, a step of Newton is applied on each leaf and overrides its value. The tree structure is untouched; only leaf values change.

2. **Structure Optimization:** During tree growth, conditions are selected according to a score that includes Newton formula components. The tree structure is impacted.

---

## Loss Functions in Scikit-Learn

| Task | Loss Function | Parameter |
|------|---------------|-----------|
| Regression | Squared Error | `loss='squared_error'` |
| Classification | Log Loss (Entropy) | `loss='log_loss'` |
| Quantile Regression | Quantile Loss | `loss='quantile'` |

---

## One-Line Summary

**Gradient boosting uses the gradient of the loss function (not just signed error) as the target for weak models, and Newton's method can optimize leaf values.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss, mean_squared_error, accuracy_score

## Classification with Log Loss

In [ ]:
# Create classification dataset
X, y = make_classification(n_samples=500, n_features=10, n_classes=2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Train size: {len(X_train)}")
print(f"Test size: {len(X_test)}")

In [ ]:
# Gradient Boosting Classifier with Log Loss
gb_clf = GradientBoostingClassifier(
    loss='log_loss',        # log loss (entropy) for classification
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

gb_clf.fit(X_train, y_train)

print("Model trained successfully")
print(f"Loss function: {gb_clf.loss}")
print(f"Number of estimators: {gb_clf.n_estimators_}")

In [ ]:
# Predict and evaluate
y_pred_train = gb_clf.predict(X_train)
y_pred_test = gb_clf.predict(X_test)

y_prob_train = gb_clf.predict_proba(X_train)[:, 1]
y_prob_test = gb_clf.predict_proba(X_test)[:, 1]

train_acc = accuracy_score(y_train, y_pred_train)
test_acc = accuracy_score(y_test, y_pred_test)
train_log_loss = log_loss(y_train, y_prob_train)
test_log_loss = log_loss(y_test, y_prob_test)

print("\nClassification Results:")
print("="*40)
print(f"Train Accuracy: {train_acc:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Train Log Loss: {train_log_loss:.4f}")
print(f"Test Log Loss: {test_log_loss:.4f}")

## Regression with Different Loss Functions

In [ ]:
# Create regression dataset
X_reg, y_reg = make_regression(n_samples=500, n_features=5, noise=10, random_state=42)
Xr_train, Xr_test, yr_train, yr_test = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

print("Regression Dataset:")
print(f"Train size: {len(Xr_train)}")
print(f"Test size: {len(Xr_test)}")

In [ ]:
# Compare different loss functions
losses = ['squared_error', 'absolute_error', 'huber', 'quantile']

print("\n" + "="*60)
print("Regression with Different Loss Functions")
print("="*60)

for loss in losses:
    gb = GradientBoostingRegressor(
        loss=loss,
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    )
    gb.fit(Xr_train, yr_train)
    mse = mean_squared_error(yr_test, gb.predict(X_test))
    print(f"Loss = {loss:<15}: Test MSE = {mse:.4f}")

## Visualizing Gradient Descent vs Newton Method

In [ ]:
# Simple visualization of gradient descent vs newton
def f(x):
    return x**4 - 4*x**2 + 4

def f_prime(x):
    return 4*x**3 - 8*x

def f_double_prime(x):
    return 12*x**2 - 8

x = np.linspace(-2.5, 2.5, 100)
y = f(x)

plt.figure(figsize=(10, 5))
plt.plot(x, y, 'b-', linewidth=2, label='Loss Function')
plt.xlabel('Parameter Value')
plt.ylabel('Loss')
plt.title('Loss Function Optimization')
plt.grid(alpha=0.3)
plt.legend()
plt.show()

print("\nNote: Newton's method uses both first and second derivatives")
print("for faster convergence compared to gradient descent.")

In [ ]:
# Compare models with different loss functions for classification
print("\n" + "="*60)
print("Classification: Different Loss Functions")
print("="*60)

gb_log = GradientBoostingClassifier(loss='log_loss', n_estimators=100, random_state=42)
gb_log.fit(X_train, y_train)

gb_exp = GradientBoostingClassifier(loss='exponential', n_estimators=100, random_state=42)
gb_exp.fit(X_train, y_train)

print(f"Log Loss (Entropy) Accuracy: {accuracy_score(y_test, gb_log.predict(X_test)):.4f}")
print(f"Exponential Loss Accuracy: {accuracy_score(y_test, gb_exp.predict(X_test)):.4f}")

In [ ]:
# Day 2 Completed
print("\n" + "="*40)
print("DAY 2 COMPLETED")
print("="*40)
print("Topics covered:")
print("- Loss functions in Gradient Boosting")
print("- Pseudo response and gradient formula")
print("- Squared error for regression (signed error)")
print("- Log loss for classification")
print("- Newton's method vs Gradient Descent")
print("- Leaf and structure optimization with Newton")
print("- Different loss functions in sklearn")
print("="*40)"
print("\nNext: Day 3 - Hyperparameter Tuning in Gradient Boosting")